# MRI 脑体素分割 - 使用Kolmogorov-Arnold网络(KAN)

这个笔记本演示如何使用KAN (Kolmogorov-Arnold Networks)替代传统MLP进行MRI脑体素分割。KAN是一种新型神经网络结构，它使用可学习的激活函数来提高性能和可解释性。

我们将使用与原始MLP相同的MRI数据，但应用KAN架构，并比较其性能。

In [5]:
# 导入KAN和其他必要的库
from fastkan import *
# from efficient_kan import *
import matplotlib.pyplot as plt
import h5py
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time
import os
import torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# 设置matplotlib显示中文
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来正常显示负号

print("所有库导入成功！")

所有库导入成功！


In [6]:
# 设置计算设备
if torch.cuda.is_available():
    device = torch.device('cuda')
    # 获取GPU信息
    gpu_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
    print(f"使用设备: {device} ({gpu_name})")
    print(f"可用GPU数量: {gpu_count}")
    !nvidia-smi
else:
    device = torch.device('cpu')
    print(f"使用设备: {device}")


使用设备: cuda (NVIDIA RTX A6000)
可用GPU数量: 1
Sun Mar  9 20:43:28 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.14              Driver Version: 550.54.14      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:4F:00.0 Off |                  Off |
| 30%   28C    P8             30W /  300W |   17971MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+------

In [9]:
# # 设置参数和路径
# output_path = '/home/jannik/Documents/brain_voxel_data/output/'  # 修改为您的数据保存路径
# export_path = '/home/jannik/Documents/brain_voxel_data/output/kan_models_binary/'  # KAN二分类模型保存路径
output_path = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output"

export_path = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/kan_models_binary/"

# 确保模型保存路径存在
os.makedirs(export_path, exist_ok=True)

# 设置训练参数
batch_size = 256  # Adam优化器的批量大小
learning_rate = 0.001  # Adam优化器的学习率

# 设置KAN网络结构 - 修改为二分类输出(1)
network_structure = [341, 512, 256, 1]  # [输入维度, 隐藏层1, 隐藏层2, 输出维度(1)]
print(f"KAN网络结构: {network_structure}")

# 网格策略设计 - 指数型增长
n_grid_extensions = 80  # 总共进行80次网格扩展
initial_grid = 3  # 从网格大小3开始
max_grid = 1000  # 最大网格大小1000

# 生成指数增长的网格序列
grid_sequence = [int(initial_grid * (max_grid/initial_grid)**(i/(n_grid_extensions-1))) 
                for i in range(n_grid_extensions)]
# 去重并排序
grid_sequence = sorted(list(set(grid_sequence)))  
# 确保最后一个网格大小不超过最大值
if grid_sequence[-1] > max_grid:
    grid_sequence[-1] = max_grid

# 将网格序列分为三个主要阶段
phase1_end = len(grid_sequence) // 3  # 第1阶段结束索引
phase2_end = 2 * len(grid_sequence) // 3  # 第2阶段结束索引

# 三个主要阶段的网格序列
phase_grid_sequences = [
    grid_sequence[:phase1_end+1],  # 第1阶段网格序列
    grid_sequence[phase1_end+1:phase2_end+1],  # 第2阶段网格序列
    grid_sequence[phase2_end+1:]  # 第3阶段网格序列
]

# 设置每个阶段的训练步数
steps_per_phase = {
    0: 30,  # 第1阶段的每个网格点训练30步
    1: 20,  # 第2阶段的每个网格点训练20步
    2: 10   # 第3阶段的每个网格点训练10步
}

# 计算总训练步数
total_steps = sum(steps_per_phase[i] * len(phase_grid_sequences[i]) for i in range(3))

print(f"参数设置完成，二分类训练, 批量大小: {batch_size}")
print(f"学习率: {learning_rate}, 总训练步数: {total_steps}")
print(f"网格扩展次数: {len(grid_sequence)}")
print(f"网格扩展序列: {grid_sequence}")
print(f"\n指数网格训练策略:")
print(f"  阶段1 (网格 {grid_sequence[0]}-{grid_sequence[phase1_end]}): 每个网格点训练 {steps_per_phase[0]} 步")
print(f"  阶段2 (网格 {grid_sequence[phase1_end+1]}-{grid_sequence[phase2_end]}): 每个网格点训练 {steps_per_phase[1]} 步")
print(f"  阶段3 (网格 {grid_sequence[phase2_end+1]}-{grid_sequence[-1]}): 每个网格点训练 {steps_per_phase[2]} 步")

KAN网络结构: [341, 512, 256, 1]
参数设置完成，二分类训练, 批量大小: 256
学习率: 0.001, 总训练步数: 1410
网格扩展次数: 70
网格扩展序列: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 23, 25, 27, 29, 31, 33, 36, 39, 42, 45, 49, 52, 56, 61, 65, 70, 76, 82, 88, 95, 102, 110, 118, 127, 137, 147, 159, 171, 184, 198, 213, 229, 247, 266, 286, 308, 331, 357, 384, 413, 445, 479, 515, 555, 597, 643, 692, 745, 802, 863, 929, 1000]

指数网格训练策略:
  阶段1 (网格 3-33): 每个网格点训练 30 步
  阶段2 (网格 36-184): 每个网格点训练 20 步
  阶段3 (网格 198-1000): 每个网格点训练 10 步


In [11]:
import numpy as np
import h5py
from sklearn.preprocessing import StandardScaler

# 文件路径
data_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat'
output_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output'
# 读取数据
f = h5py.File(data_path, 'r')
arrays = {k: np.array(v) for k, v in f.items()}
f.close()

train_data = arrays['data'].transpose()
train_region = arrays['region'].transpose()
prob_idx = arrays['prob_idx'].transpose()

# 数据分割 - 38号作为验证集，其余作为完整训练集
train_indices = np.where(prob_idx != 38)[0]
x_train_full = train_data[train_indices, :]
y_train_full = train_region[train_indices, :]

val_indices = np.where(prob_idx == 38)[0]
val_data = train_data[val_indices, :]
val_label = train_region[val_indices, :]

# 标准化
scaler = StandardScaler()
scaler.fit(x_train_full)
x_train_full = scaler.transform(x_train_full)
val_data = scaler.transform(val_data)

# 保存数据
np.save(f"{output_path}/x_train_full.npy", x_train_full)
np.save(f"{output_path}/y_train_full.npy", y_train_full)
np.save(f"{output_path}/val_data.npy", val_data)
np.save(f"{output_path}/val_label.npy", val_label)

print(f"Data saved to {output_path}")

Data saved to /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output


In [13]:
import numpy as np
import os

# 文件路径
input_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/full'
output_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/by_label'

# 创建输出目录（如果不存在）
os.makedirs(output_path, exist_ok=True)

# 加载数据
x_train_full = np.load(f"{input_path}/x_train_full.npy")
y_train_full = np.load(f"{input_path}/y_train_full.npy")

print(f"加载数据: X shape = {x_train_full.shape}, Y shape = {y_train_full.shape}")

# 确认标签是one-hot编码
num_labels = y_train_full.shape[1]
if num_labels != 102:
    print(f"警告: 标签数量 {num_labels} 与预期的 102 不符")

# 提取每个标签的体素
for label_idx in range(num_labels):
    # 找到标签为label_idx的所有体素索引
    # 在one-hot编码中，标签位置的值为1
    voxel_indices = np.where(y_train_full[:, label_idx] == 1)[0]
    
    if len(voxel_indices) == 0:
        print(f"标签 {label_idx} 没有对应的体素")
        continue
    
    # 提取这些体素的biosignature数据
    label_voxels = x_train_full[voxel_indices]
    
    # 保存到文件，文件名中包含体素数量信息
    voxel_count = len(voxel_indices)
    output_file = f"{output_path}/label_{label_idx}_count_{voxel_count}_voxels.npy"
    np.save(output_file, label_voxels)
    
    print(f"标签 {label_idx}: 提取了 {voxel_count} 个体素，形状 {label_voxels.shape}，保存到 {output_file}")

print("所有标签的体素数据提取完成")

# 验证提取结果
total_voxels = 0
empty_labels = []
non_empty_labels = []

for label_idx in range(num_labels):
    # 搜索匹配的文件名模式
    label_pattern = f"label_{label_idx}_count_"
    matching_files = [f for f in os.listdir(output_path) if f.startswith(label_pattern)]
    
    if not matching_files:
        empty_labels.append(label_idx)
        print(f"标签 {label_idx}: 没有包含任何体素")
        continue
        
    try:
        label_file = os.path.join(output_path, matching_files[0])
        label_data = np.load(label_file)
        voxel_count = len(label_data)
        total_voxels += voxel_count
        non_empty_labels.append((label_idx, voxel_count))
        print(f"标签 {label_idx}: 包含 {voxel_count} 个体素")
    except Exception as e:
        print(f"读取标签 {label_idx} 数据时出错: {e}")

# 输出结果统计
print(f"原始体素总数: {len(x_train_full)}, 提取后体素总数: {total_voxels}")
if len(x_train_full) == total_voxels:
    print("✓ 提取结果验证通过: 所有体素数量匹配")
else:
    print("✗ 提取结果验证失败: 体素数量不匹配")

# 更严格的验证: 检查每个标签的每个体素数据是否与原始数据一致
print("\n开始进行严格的数据一致性验证...")
validation_passed = True
mismatch_count = 0

for label_idx in range(num_labels):
    # 从原始数据中找出标签为label_idx的体素索引（该标签位置值为1的所有体素）
    orig_indices = np.where(y_train_full[:, label_idx] == 1)[0]
    
    # 如果没有体素属于这个标签，跳过验证
    if len(orig_indices) == 0:
        if label_idx not in empty_labels:
            print(f"警告: 标签 {label_idx} 在原始数据中没有体素，但未记录为空标签")
            validation_passed = False
        continue
        
    # 加载提取后的数据
    matching_files = [f for f in os.listdir(output_path) if f.startswith(f"label_{label_idx}_count_")]
    if not matching_files:
        print(f"错误: 标签 {label_idx} 应有数据但未找到对应文件")
        validation_passed = False
        continue
        
    extracted_data = np.load(os.path.join(output_path, matching_files[0]))
    
    # 检查数量是否匹配
    if len(orig_indices) != len(extracted_data):
        print(f"错误: 标签 {label_idx} 体素数量不匹配 - 原始: {len(orig_indices)}, 提取后: {len(extracted_data)}")
        validation_passed = False
        continue
    
    # 逐个验证体素数据一致性
    for idx, orig_idx in enumerate(orig_indices):
        # 确认原始数据中该索引处的体素确实属于当前标签
        if y_train_full[orig_idx, label_idx] != 1:
            print(f"错误: 索引 {orig_idx} 的体素标签值不为1，但被包含在标签 {label_idx} 的索引中")
            validation_passed = False
            continue
            
        # 比较原始数据和提取后数据
        orig_voxel = x_train_full[orig_idx]
        extracted_voxel = extracted_data[idx]
        
        if not np.array_equal(orig_voxel, extracted_voxel):
            mismatch_count += 1
            if mismatch_count <= 5:  # 仅显示前5个不匹配的详细信息
                print(f"数据不匹配: 标签 {label_idx}, 原始索引 {orig_idx}, 提取后索引 {idx}")
                # 显示不匹配的差异
                diff = orig_voxel - extracted_voxel
                non_zero_diff = np.count_nonzero(diff)
                if non_zero_diff > 0:
                    print(f"   不匹配元素数: {non_zero_diff}, 最大差异: {np.max(np.abs(diff))}")
            validation_passed = False
    
    # 每处理10个标签打印一次进度
    if label_idx % 10 == 0 and label_idx > 0:
        print(f"已验证 {label_idx}/{num_labels} 个标签...")

if validation_passed:
    print("\n✓ 严格验证通过: 所有体素数据与原始数据完全一致!")
else:
    print(f"\n✗ 严格验证失败: 发现 {mismatch_count} 个体素数据不匹配!")

# 输出空标签和非空标签的统计信息
print(f"\n空标签数量: {len(empty_labels)}")
if empty_labels:
    print(f"空标签索引: {empty_labels}")

print(f"\n非空标签数量: {len(non_empty_labels)}")
non_empty_labels.sort(key=lambda x: x[1], reverse=True)
print("\n体素数量前10的标签:")
for idx, (label, count) in enumerate(non_empty_labels[:10], 1):
    print(f"{idx}. 标签 {label}: {count} 个体素")

# 创建一个索引文件，便于后续处理
index_file = f"{output_path}/label_index.txt"
with open(index_file, 'w') as f:
    f.write("label_id,voxel_count,filename\n")
    for label_idx in range(num_labels):
        matching_files = [file for file in os.listdir(output_path) if file.startswith(f"label_{label_idx}_count_")]
        if matching_files:
            filename = matching_files[0]
            count = int(filename.split("_count_")[1].split("_")[0])
            f.write(f"{label_idx},{count},{filename}\n")
        else:
            f.write(f"{label_idx},0,\n")
            
print(f"\n已创建标签索引文件: {index_file}")

加载数据: X shape = (6795317, 341), Y shape = (6795317, 102)
标签 0: 提取了 207692 个体素，形状 (207692, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/by_label/label_0_count_207692_voxels.npy
标签 1: 提取了 16143 个体素，形状 (16143, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/by_label/label_1_count_16143_voxels.npy
标签 2: 提取了 16483 个体素，形状 (16483, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/by_label/label_2_count_16483_voxels.npy
标签 3: 提取了 186213 个体素，形状 (186213, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/by_label/label_3_count_186213_voxels.npy
标签 4: 提取了 170589 个体素，形状 (170589, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/by_label/label_4_count_170589_voxels.npy
标签 5: 提取了 68810 个体素，形状 (68810, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/by_label/label_5_count_68810_voxels.np

In [14]:
# 创建val_per_label数据集
import numpy as np
import os

# 文件路径
# input_path = '/home/jannik/Documents/brain_voxel_data/output'
# output_path = '/home/jannik/Documents/brain_voxel_data/output/val_by_label'
input_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/full'
output_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_by_label'

# 创建输出目录（如果不存在）
os.makedirs(output_path, exist_ok=True)

# 加载验证数据
val_data = np.load(f"{input_path}/val_data.npy")
val_label = np.load(f"{input_path}/val_label.npy")

print(f"加载验证数据: X shape = {val_data.shape}, Y shape = {val_label.shape}")

# 确认标签是one-hot编码
num_labels = val_label.shape[1]
if num_labels != 102:
    print(f"警告: 标签数量 {num_labels} 与预期的 102 不符")

# 提取每个标签的体素
for label_idx in range(num_labels):
    # 找到标签为label_idx的所有体素索引
    # 在one-hot编码中，标签位置的值为1
    voxel_indices = np.where(val_label[:, label_idx] == 1)[0]
    
    if len(voxel_indices) == 0:
        print(f"标签 {label_idx} 没有对应的体素")
        continue
    
    # 提取这些体素的biosignature数据
    label_voxels = val_data[voxel_indices]
    
    # 保存到文件，文件名中包含体素数量信息
    voxel_count = len(voxel_indices)
    output_file = f"{output_path}/label_{label_idx}_count_{voxel_count}_voxels.npy"
    np.save(output_file, label_voxels)
    
    print(f"标签 {label_idx}: 提取了 {voxel_count} 个体素，形状 {label_voxels.shape}，保存到 {output_file}")

print("所有验证集标签的体素数据提取完成")

# 验证提取结果
total_voxels = 0
empty_labels = []
non_empty_labels = []

for label_idx in range(num_labels):
    # 搜索匹配的文件名模式
    label_pattern = f"label_{label_idx}_count_"
    matching_files = [f for f in os.listdir(output_path) if f.startswith(label_pattern)]
    
    if not matching_files:
        empty_labels.append(label_idx)
        print(f"标签 {label_idx}: 没有包含任何体素")
        continue
        
    try:
        label_file = os.path.join(output_path, matching_files[0])
        label_data = np.load(label_file)
        voxel_count = len(label_data)
        total_voxels += voxel_count
        non_empty_labels.append((label_idx, voxel_count))
        print(f"标签 {label_idx}: 包含 {voxel_count} 个体素")
    except Exception as e:
        print(f"读取标签 {label_idx} 数据时出错: {e}")

# 输出结果统计
print(f"原始验证集体素总数: {len(val_data)}, 提取后体素总数: {total_voxels}")
if len(val_data) == total_voxels:
    print("✓ 提取结果验证通过: 所有体素数量匹配")
else:
    print("✗ 提取结果验证失败: 体素数量不匹配")

# 更严格的验证: 检查每个标签的每个体素数据是否与原始数据一致
print("\n开始进行严格的数据一致性验证...")
validation_passed = True
mismatch_count = 0

for label_idx in range(num_labels):
    # 从原始数据中找出标签为label_idx的体素索引（该标签位置值为1的所有体素）
    orig_indices = np.where(val_label[:, label_idx] == 1)[0]
    
    # 如果没有体素属于这个标签，跳过验证
    if len(orig_indices) == 0:
        if label_idx not in empty_labels:
            print(f"警告: 标签 {label_idx} 在原始数据中没有体素，但未记录为空标签")
            validation_passed = False
        continue
        
    # 加载提取后的数据
    matching_files = [f for f in os.listdir(output_path) if f.startswith(f"label_{label_idx}_count_")]
    if not matching_files:
        print(f"错误: 标签 {label_idx} 应有数据但未找到对应文件")
        validation_passed = False
        continue
        
    extracted_data = np.load(os.path.join(output_path, matching_files[0]))
    
    # 检查数量是否匹配
    if len(orig_indices) != len(extracted_data):
        print(f"错误: 标签 {label_idx} 体素数量不匹配 - 原始: {len(orig_indices)}, 提取后: {len(extracted_data)}")
        validation_passed = False
        continue
    
    # 逐个验证体素数据一致性
    for idx, orig_idx in enumerate(orig_indices):
        # 确认原始数据中该索引处的体素确实属于当前标签
        if val_label[orig_idx, label_idx] != 1:
            print(f"错误: 索引 {orig_idx} 的体素标签值不为1，但被包含在标签 {label_idx} 的索引中")
            validation_passed = False
            continue
            
        # 比较原始数据和提取后数据
        orig_voxel = val_data[orig_idx]
        extracted_voxel = extracted_data[idx]
        
        if not np.array_equal(orig_voxel, extracted_voxel):
            mismatch_count += 1
            if mismatch_count <= 5:  # 仅显示前5个不匹配的详细信息
                print(f"数据不匹配: 标签 {label_idx}, 原始索引 {orig_idx}, 提取后索引 {idx}")
                # 显示不匹配的差异
                diff = orig_voxel - extracted_voxel
                non_zero_diff = np.count_nonzero(diff)
                if non_zero_diff > 0:
                    print(f"   不匹配元素数: {non_zero_diff}, 最大差异: {np.max(np.abs(diff))}")
            validation_passed = False
    
    # 每处理10个标签打印一次进度
    if label_idx % 10 == 0 and label_idx > 0:
        print(f"已验证 {label_idx}/{num_labels} 个标签...")

if validation_passed:
    print("\n✓ 严格验证通过: 所有体素数据与原始数据完全一致!")
else:
    print(f"\n✗ 严格验证失败: 发现 {mismatch_count} 个体素数据不匹配!")

# 输出空标签和非空标签的统计信息
print(f"\n空标签数量: {len(empty_labels)}")
if empty_labels:
    print(f"空标签索引: {empty_labels}")

print(f"\n非空标签数量: {len(non_empty_labels)}")
non_empty_labels.sort(key=lambda x: x[1], reverse=True)
print("\n体素数量前10的标签:")
for idx, (label, count) in enumerate(non_empty_labels[:10], 1):
    print(f"{idx}. 标签 {label}: {count} 个体素")

# 创建一个索引文件，便于后续处理
index_file = f"{output_path}/val_label_index.txt"
with open(index_file, 'w') as f:
    f.write("label_id,voxel_count,filename\n")
    for label_idx in range(num_labels):
        matching_files = [file for file in os.listdir(output_path) if file.startswith(f"label_{label_idx}_count_")]
        if matching_files:
            filename = matching_files[0]
            count = int(filename.split("_count_")[1].split("_")[0])
            f.write(f"{label_idx},{count},{filename}\n")
        else:
            f.write(f"{label_idx},0,\n")
            
print(f"\n已创建验证集标签索引文件: {index_file}")

加载验证数据: X shape = (173383, 341), Y shape = (173383, 102)
标签 0: 提取了 2477 个体素，形状 (2477, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_by_label/label_0_count_2477_voxels.npy
标签 1: 提取了 363 个体素，形状 (363, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_by_label/label_1_count_363_voxels.npy
标签 2: 提取了 333 个体素，形状 (333, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_by_label/label_2_count_333_voxels.npy
标签 3: 提取了 4599 个体素，形状 (4599, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_by_label/label_3_count_4599_voxels.npy
标签 4: 提取了 4944 个体素，形状 (4944, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_by_label/label_4_count_4944_voxels.npy
标签 5: 提取了 1739 个体素，形状 (1739, 341)，保存到 /home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_by_label/label_5_count_1739_voxels.npy
标签 6: 提

In [7]:
import numpy as np
import os
import torch
import time
import glob
import random
from tqdm import tqdm
from sklearn.utils import shuffle

# 定义全局配置类
class DataConfig:
    def __init__(self):
        # 数据路径配置
        self.output_path = '/home/jannik/Documents/brain_voxel_data/output'
        self.train_label_dir = '/home/jannik/Documents/brain_voxel_data/output/by_label'
        self.val_label_dir = '/home/jannik/Documents/brain_voxel_data/output/val_by_label'
        
        # 设备配置
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 数据集配置
        self.num_labels = 102
        self.feature_dim = 341
        self.negative_ratio = 3  # 负样本与正样本的比例
        self.val_size = 1000     # 每个标签验证集样本上限
        
        # 增强配置
        self.enable_augmentation = True
        self.min_voxel_threshold = 5000  # 低于此值的类别将被视为小类别
        self.noise_scale = 0.01          # 噪声强度系数
        self.enable_noise_augmentation = True
        self.enable_oversampling = True
        self.oversampling_ratio = 2.0    # 过采样倍数
        
        # 可以添加高斯噪声的特征索引范围
        self.noise_applicable_features = [
            (0, 15),     # 扩散MRI相关指标 (0-14)
            (225, 229)   # MT & CEST信号 (225-228)
        ]

# 数据增强函数
def apply_gaussian_noise(data, noise_scale=0.01, applicable_features=None):
    """
    对特定特征维度应用高斯噪声
    
    参数:
        data: 输入数据，形状为(n_samples, n_features)
        noise_scale: 噪声强度系数
        applicable_features: 可应用噪声的特征索引范围列表，如[(0,15), (225,229)]
    
    返回:
        增强后的数据
    """
    # 复制原始数据，避免修改原数据
    augmented_data = data.copy()
    
    # 如果没有指定可应用噪声的特征，默认对所有特征应用
    if applicable_features is None:
        # 计算数据标准差
        std_dev = np.std(data, axis=0)
        # 生成随机噪声
        noise = np.random.normal(0, noise_scale * std_dev, data.shape)
        # 应用噪声
        augmented_data = data + noise
    else:
        # 仅对指定的特征索引范围应用噪声
        for start_idx, end_idx in applicable_features:
            # 选择特征子集
            feature_subset = data[:, start_idx:end_idx]
            # 计算该子集的标准差
            std_dev = np.std(feature_subset, axis=0)
            # 生成随机噪声
            noise = np.random.normal(0, noise_scale * std_dev, feature_subset.shape)
            # 应用噪声
            augmented_data[:, start_idx:end_idx] = feature_subset + noise
    
    return augmented_data

# 辅助函数：读取标签索引文件
def load_label_index(index_file):
    label_info = {}
    with open(index_file, 'r') as f:
        # 跳过表头
        next(f)
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                label_id = int(parts[0])
                voxel_count = int(parts[1])
                filename = parts[2] if parts[2] else None
                label_info[label_id] = {'count': voxel_count, 'filename': filename}
    return label_info

# 数据管理器类
class BrainVoxelDataManager:
    def __init__(self, config=None):
        """
        初始化数据管理器
        
        参数:
            config: 配置对象，如果为None则使用默认配置
        """
        # 使用提供的配置或创建默认配置
        self.config = config if config else DataConfig()
        
        # 读取标签索引文件
        train_index_file = os.path.join(self.config.train_label_dir, "label_index.txt")
        val_index_file = os.path.join(self.config.val_label_dir, "val_label_index.txt")
        
        if not os.path.exists(train_index_file):
            raise FileNotFoundError(f"训练标签索引文件不存在: {train_index_file}")
        if not os.path.exists(val_index_file):
            raise FileNotFoundError(f"验证标签索引文件不存在: {val_index_file}")
        
        self.train_label_info = load_label_index(train_index_file)
        self.val_label_info = load_label_index(val_index_file)
        
        # 筛选有效标签（有体素数据的标签）
        self.valid_labels = [label_id for label_id, info in self.train_label_info.items() 
                            if info['count'] > 0]
        
        print(f"数据管理器初始化完成: 找到 {len(self.valid_labels)} 个有效标签")
        
        # 打印增强配置信息
        if self.config.enable_augmentation:
            print(f"启用小类别增强, 阈值: {self.config.min_voxel_threshold} 体素")
            if self.config.enable_noise_augmentation:
                print(f"  - 启用噪声增强, 噪声系数: {self.config.noise_scale}")
            if self.config.enable_oversampling:
                print(f"  - 启用过采样, 过采样比例: {self.config.oversampling_ratio}x")
        
        # 验证配置
        self._validate_config()
    
    def _validate_config(self):
        """验证配置的有效性"""
        if not os.path.exists(self.config.train_label_dir):
            raise FileNotFoundError(f"训练数据目录不存在: {self.config.train_label_dir}")
        if not os.path.exists(self.config.val_label_dir):
            raise FileNotFoundError(f"验证数据目录不存在: {self.config.val_label_dir}")
    
    def get_label_file_path(self, label_id, is_validation=False):
        """获取指定标签ID的数据文件路径"""
        if is_validation:
            pattern = os.path.join(self.config.val_label_dir, f"label_{label_id}_count_*_voxels.npy")
        else:
            pattern = os.path.join(self.config.train_label_dir, f"label_{label_id}_count_*_voxels.npy")
        
        matches = glob.glob(pattern)
        return matches[0] if matches else None
    
    def get_all_valid_labels(self):
        """返回所有有效的标签ID列表"""
        return self.valid_labels.copy()
    
    def get_label_info(self, label_id):
        """获取指定标签的信息"""
        train_info = self.train_label_info.get(label_id, {'count': 0, 'filename': None})
        val_info = self.val_label_info.get(label_id, {'count': 0, 'filename': None})
        
        return {
            'label_id': label_id,
            'train_count': train_info['count'],
            'val_count': val_info['count'],
            'train_file': train_info['filename'],
            'val_file': val_info['filename']
        }
    
    def get_dataset_for_label(self, target_label_id, verbose=True):
        """
        为特定标签创建完整的数据集字典，包含训练数据和验证数据
        
        参数:
            target_label_id: 目标类别的标签ID
            verbose: 是否打印详细信息
            
        返回:
            full_dataset: 包含训练和验证数据的字典
        """
        # 确保标签ID有效
        if target_label_id not in self.valid_labels and target_label_id < self.config.num_labels:
            if verbose:
                print(f"警告: 标签 {target_label_id} 没有训练数据，将返回空数据集")
            
            # 返回空数据集
            return {
                'train_data': np.array([]).reshape(0, self.config.feature_dim),
                'train_label': np.array([]).reshape(0, 1),
                'test_input': torch.tensor([], device=self.config.device),
                'test_label': torch.tensor([], device=self.config.device),
                'label_id': target_label_id
            }
        
        if verbose:
            print(f"为标签 {target_label_id} 创建数据集...")
        
        # 1. 加载目标类别的所有训练体素（正样本）
        target_file = self.get_label_file_path(target_label_id)
        if not target_file:
            raise ValueError(f"标签 {target_label_id} 没有对应的训练数据文件")
        
        positive_samples = np.load(target_file)
        num_positive = len(positive_samples)
        original_num_positive = num_positive  # 记录原始正样本数量用于报告
        
        if verbose:
            print(f"  原始正样本数量: {num_positive}")
        
        # 检查是否需要对小类别应用增强
        need_augmentation = (self.config.enable_augmentation and 
                            num_positive < self.config.min_voxel_threshold)
        
        # 如果需要增强：先复制体素，然后对复制的体素添加噪声
        if need_augmentation:
            # 保存原始样本
            augmented_samples = []
            
            # 应用过采样并添加噪声
            if self.config.enable_oversampling:
                # 计算需要复制的次数（过采样倍数-1，因为原始样本已经有1倍）
                copies_needed = int(self.config.oversampling_ratio - 1)
                if verbose:
                    print(f"  复制体素 {copies_needed} 次")
                
                for i in range(copies_needed):
                    # 复制原始样本
                    copied_samples = positive_samples.copy()
                    
                    # 如果启用噪声增强，对复制的样本添加噪声
                    if self.config.enable_noise_augmentation:
                        copied_samples = apply_gaussian_noise(
                            copied_samples, 
                            self.config.noise_scale,
                            self.config.noise_applicable_features
                        )
                        if verbose:
                            print(f"  第 {i+1} 份复制体素已添加噪声 (噪声系数: {self.config.noise_scale})")
                    
                    # 将处理后的复制样本添加到增强样本列表中
                    augmented_samples.append(copied_samples)
            
            # 如果有增强样本，合并到原始样本中
            if augmented_samples:
                augmented_samples = np.vstack(augmented_samples)
                positive_samples = np.vstack([positive_samples, augmented_samples])
                if verbose:
                    print(f"  增强后正样本数量: {len(positive_samples)}")
        
        num_positive = len(positive_samples)
        
        # 2. 计算需要的负样本数量
        num_negative = num_positive * self.config.negative_ratio
        
        # 3. 从其他类别中随机抽取负样本
        other_labels = [l for l in self.valid_labels if l != target_label_id]
        
        # 负样本集合
        negative_samples = []
        negative_count = 0
        
        # 随机打乱其他标签顺序
        random.shuffle(other_labels)
        
        # 从其他类别中抽取负样本，直到达到所需数量
        for label_id in other_labels:
            if negative_count >= num_negative:
                break
                
            label_file = self.get_label_file_path(label_id)
            if not label_file:
                continue
                
            label_samples = np.load(label_file)
            
            # 如果当前类别的样本太多，随机抽取一部分
            samples_needed = min(len(label_samples), num_negative - negative_count)
            if samples_needed < len(label_samples):
                indices = np.random.choice(len(label_samples), samples_needed, replace=False)
                sampled = label_samples[indices]
            else:
                sampled = label_samples
                
            negative_samples.append(sampled)
            negative_count += len(sampled)
        
        # 合并所有负样本
        if negative_samples:
            negative_samples = np.vstack(negative_samples)
            # 如果收集到的负样本超过需求，再次随机抽取
            if len(negative_samples) > num_negative:
                indices = np.random.choice(len(negative_samples), num_negative, replace=False)
                negative_samples = negative_samples[indices]
        else:
            negative_samples = np.array([]).reshape(0, positive_samples.shape[1])
            
        if verbose:
            print(f"  负样本数量: {len(negative_samples)}")
        
        # 4. 创建特征数据和标签
        X_train = np.vstack([positive_samples, negative_samples])
        
        # 创建标签: 正样本为1，负样本为0
        y_positive = np.ones((num_positive, 1))
        y_negative = np.zeros((len(negative_samples), 1))
        y_train = np.vstack([y_positive, y_negative])
        
        # 5. 随机打乱数据 - 使用相同的随机状态确保X和y保持一一对应关系
        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_train = X_train[indices]
        y_train = y_train[indices]
        
        if verbose:
            print(f"  总训练样本: {len(X_train)}, 特征维度: {X_train.shape[1]}")
            print(f"  正样本比例: {np.mean(y_train):.4f}")
        
        # 如果应用了增强，显示增强汇总
        if verbose and need_augmentation and (self.config.enable_noise_augmentation or 
                               self.config.enable_oversampling):
            print(f"  增强汇总:")
            print(f"    - 原始正样本: {original_num_positive}")
            print(f"    - 增强后正样本: {num_positive}")
            print(f"    - 增强倍数: {num_positive/original_num_positive:.2f}x")
        
        # 6. 加载对应的验证数据（同样的标签）
        X_val = None
        y_val = None
        
        val_file = self.get_label_file_path(target_label_id, is_validation=True)
        if val_file:
            # 加载验证集正样本
            val_positive_samples = np.load(val_file)
            val_num_positive = len(val_positive_samples)
            
            # 如果验证集太大，随机抽样
            if val_num_positive > self.config.val_size:
                indices = np.random.choice(val_num_positive, self.config.val_size, replace=False)
                val_positive_samples = val_positive_samples[indices]
                val_num_positive = len(val_positive_samples)
            
            # 从其他类别抽取验证集负样本
            val_num_negative = val_num_positive * self.config.negative_ratio
            val_negative_samples = []
            val_negative_count = 0
            
            # 随机打乱其他标签
            random.shuffle(other_labels)
            
            # 从其他类别抽取验证集负样本
            for label_id in other_labels:
                if val_negative_count >= val_num_negative:
                    break
                    
                val_label_file = self.get_label_file_path(label_id, is_validation=True)
                if not val_label_file:
                    continue
                    
                val_label_samples = np.load(val_label_file)
                
                # 如果当前类别样本太多，随机抽样
                samples_needed = min(len(val_label_samples), val_num_negative - val_negative_count)
                if samples_needed < len(val_label_samples):
                    indices = np.random.choice(len(val_label_samples), samples_needed, replace=False)
                    sampled = val_label_samples[indices]
                else:
                    sampled = val_label_samples
                    
                val_negative_samples.append(sampled)
                val_negative_count += len(sampled)
            
            # 合并验证集负样本
            if val_negative_samples:
                val_negative_samples = np.vstack(val_negative_samples)
                # 如果超过需求，再次随机抽样
                if len(val_negative_samples) > val_num_negative:
                    indices = np.random.choice(len(val_negative_samples), val_num_negative, replace=False)
                    val_negative_samples = val_negative_samples[indices]
            else:
                val_negative_samples = np.array([]).reshape(0, val_positive_samples.shape[1])
            
            # 合并验证集
            X_val = np.vstack([val_positive_samples, val_negative_samples])
            
            # 创建验证集标签
            y_val_positive = np.ones((val_num_positive, 1))
            y_val_negative = np.zeros((len(val_negative_samples), 1))
            y_val = np.vstack([y_val_positive, y_val_negative])
            
            # 随机打乱验证集
            indices = np.arange(X_val.shape[0])
            np.random.shuffle(indices)
            X_val = X_val[indices]
            y_val = y_val[indices]
            
            if verbose:
                print(f"  验证集: {len(X_val)} 个样本, 正样本比例: {np.mean(y_val):.4f}")
        else:
            if verbose:
                print(f"  警告: 标签 {target_label_id} 没有验证数据，将使用空验证集")
            X_val = np.array([]).reshape(0, self.config.feature_dim)
            y_val = np.array([]).reshape(0, 1)
        
        # 7. 转换验证数据为PyTorch张量
        test_input = torch.from_numpy(X_val).float().to(self.config.device) if len(X_val) > 0 else torch.tensor([], device=self.config.device)
        test_label = torch.from_numpy(y_val).float().to(self.config.device) if len(y_val) > 0 else torch.tensor([], device=self.config.device)
        
        # 8. 创建完整数据集字典（保留NumPy格式的训练数据）
        full_dataset = {
            'train_data': X_train,         # NumPy格式训练特征
            'train_label': y_train,        # NumPy格式训练标签
            'test_input': test_input,      # PyTorch张量验证特征
            'test_label': test_label,      # PyTorch张量验证标签
            'label_id': target_label_id    # 标签ID
        }
        
        return full_dataset
    
    def get_train_batch(self, dataset, batch_size):
        """
        从数据集中随机采样训练批次
        
        参数:
            dataset: 数据集字典
            batch_size: 批次大小
        
        返回:
            X_batch_tensor: 特征批次张量
            y_batch_tensor: 标签批次张量
        """
        train_data = dataset['train_data']
        train_label = dataset['train_label']
        
        # 确保批次大小不超过数据集大小
        batch_size = min(batch_size, train_data.shape[0])
        
        # 随机选择索引
        indices = np.random.choice(train_data.shape[0], batch_size, replace=False)
        
        # 提取批次数据
        X_batch = train_data[indices]
        y_batch = train_label[indices]
        
        # 转换为PyTorch张量
        X_batch_tensor = torch.from_numpy(X_batch).float().to(self.config.device)
        y_batch_tensor = torch.from_numpy(y_batch).float().to(self.config.device)
        
        return X_batch_tensor, y_batch_tensor
    
    def enable_augmentation(self, enable=True):
        """启用或禁用数据增强"""
        self.config.enable_augmentation = enable
        return self
    
    def set_noise_scale(self, scale):
        """设置噪声尺度"""
        self.config.noise_scale = scale
        return self
    
    def set_oversampling_ratio(self, ratio):
        """设置过采样比例"""
        self.config.oversampling_ratio = ratio
        return self
    
    def set_negative_ratio(self, ratio):
        """设置负样本比例"""
        self.config.negative_ratio = ratio
        return self


# 创建一个简单的模型训练器类
class BrainVoxelModelTrainer:
    def __init__(self, data_manager, model_constructor, export_path, device=None):
        """
        初始化训练器
        
        参数:
            data_manager: 数据管理器实例
            model_constructor: 创建模型的函数
            export_path: 模型导出路径
            device: 计算设备
        """
        self.data_manager = data_manager
        self.model_constructor = model_constructor
        self.export_path = export_path
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 创建导出目录
        os.makedirs(self.export_path, exist_ok=True)
    
    def train_model_for_label(self, label_id, training_config):
        """
        为指定标签训练模型
        
        参数:
            label_id: 目标标签ID
            training_config: 训练配置字典
            
        返回:
            训练好的模型和训练历史
        """
        print(f"\n{'='*50}")
        print(f"训练标签 {label_id} 的模型")
        print(f"{'='*50}")
        
        # 获取该标签的数据集
        dataset = self.data_manager.get_dataset_for_label(label_id)
        
        # 检查数据是否足够
        if dataset['train_data'].shape[0] < training_config.get('min_samples', 100):
            print(f"警告: 标签 {label_id} 的训练样本太少 ({dataset['train_data'].shape[0]}), 跳过训练")
            return None, None
        
        # 创建模型
        model = self.model_constructor(training_config)
        model = model.to(self.device)
        
        print(f"创建模型完成，开始训练...")
        
        # 在这里实现模型训练逻辑
        # ...
        
        # 示例：训练循环
        for epoch in range(training_config.get('epochs', 10)):
            # 训练代码...
            print(f"Epoch {epoch+1}/{training_config.get('epochs', 10)}")
            
            # 使用数据管理器获取训练批次
            X_batch, y_batch = self.data_manager.get_train_batch(dataset, training_config.get('batch_size', 64))
            
            # 模型训练步骤...
            
        return model, {"label_id": label_id, "history": []}



# 创建数据配置
config = DataConfig()

config.enable_augmentation = True

# 初始化数据管理器
data_manager = BrainVoxelDataManager(config)

# 获取所有有效标签
valid_labels = data_manager.get_all_valid_labels()
print(f"有效标签数量: {len(valid_labels)}")

# # 示例：处理前5个标签
# for i, label_id in enumerate(valid_labels[:5]):
#     print(f"\n处理标签 {label_id} ({i+1}/5)...")
    
#     # 获取标签信息
#     label_info = data_manager.get_label_info(label_id)
#     print(f"标签 {label_id} 信息:")
#     print(f"  训练样本数: {label_info['train_count']}")
#     print(f"  验证样本数: {label_info['val_count']}")
    
#     # 获取数据集
#     dataset = data_manager.get_dataset_for_label(label_id)
#     print(f"数据集大小:")
#     print(f"  训练特征: {dataset['train_data'].shape}")
#     print(f"  训练标签: {dataset['train_label'].shape}")
#     print(f"  验证特征: {dataset['test_input'].shape}")
#     print(f"  验证标签: {dataset['test_label'].shape}")
    
#     # 获取训练批次示例
#     batch_size = 32
#     X_batch, y_batch = data_manager.get_train_batch(dataset, batch_size)
#     print(f"训练批次大小: {X_batch.shape}, {y_batch.shape}")
    
#    # 这里可以传入模型和开始训练


label_id = 5

# 获取标签信息
label_info = data_manager.get_label_info(label_id)
print(f"标签 {label_id} 信息:")
print(f"  训练样本数: {label_info['train_count']}")
print(f"  验证样本数: {label_info['val_count']}")

# 获取数据集
dataset = data_manager.get_dataset_for_label(label_id)
print(f"数据集大小:")
print(f"  训练特征: {dataset['train_data'].shape}")
print(f"  训练标签: {dataset['train_label'].shape}")
print(f"  验证特征: {dataset['test_input'].shape}")
print(f"  验证标签: {dataset['test_label'].shape}")

# 获取训练批次示例
batch_size = 32
X_batch, y_batch = data_manager.get_train_batch(dataset, batch_size)
print(f"训练批次大小: {X_batch.shape}, {y_batch.shape}")

数据管理器初始化完成: 找到 101 个有效标签
启用小类别增强, 阈值: 5000 体素
  - 启用噪声增强, 噪声系数: 0.01
  - 启用过采样, 过采样比例: 2.0x
有效标签数量: 101
标签 5 信息:
  训练样本数: 68810
  验证样本数: 1739
为标签 5 创建数据集...
  原始正样本数量: 68810
  负样本数量: 206430
  总训练样本: 275240, 特征维度: 341
  正样本比例: 0.2500
  验证集: 4000 个样本, 正样本比例: 0.2500
数据集大小:
  训练特征: (275240, 341)
  训练标签: (275240, 1)
  验证特征: torch.Size([4000, 341])
  验证标签: torch.Size([4000, 1])
训练批次大小: torch.Size([32, 341]), torch.Size([32, 1])


In [8]:
# 训练所有类别的二分类模型循环
import numpy as np
import os
import torch
import torch.nn.functional as F
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
from datetime import datetime

# 辅助函数:获取模型当前的网格大小
def get_grid_size(model):
    return model.layers[0].rbf.num_grids

# 二分类评估函数
def calculate_binary_metrics(y_true, y_pred, threshold=0.5):
    """计算二分类模型的性能指标"""
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.cpu().numpy()
    if isinstance(y_pred, torch.Tensor):
        y_pred = y_pred.cpu().numpy()
    
    # 将预测转换为二分类
    y_pred_binary = (y_pred > threshold).astype(np.float32)
    
    # 计算准确率
    accuracy = np.mean(y_pred_binary == y_true)
    
    # 计算其他可能指标（例如精确度、召回率等）
    true_positives = np.sum((y_true == 1) & (y_pred_binary == 1))
    false_positives = np.sum((y_true == 0) & (y_pred_binary == 1))
    true_negatives = np.sum((y_true == 0) & (y_pred_binary == 0))
    false_negatives = np.sum((y_true == 1) & (y_pred_binary == 0))
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': true_positives,
        'fp': false_positives,
        'tn': true_negatives,
        'fn': false_negatives
    }

# 初始化采样器
print("初始化平衡采样器...")
aug_config = AugmentationConfig()
aug_config.enable_augmentation = True
aug_config.min_voxel_threshold = 5000  # 低于5000个体素的类别将被增强
aug_config.enable_noise_augmentation = True
aug_config.noise_scale = 0.01
aug_config.enable_oversampling = True
aug_config.oversampling_ratio = 2.0

sampler = BalancedVoxelSampler(
    label_dir=label_dir,
    negative_ratio=3,  # K=3，负/正=3:1
    val_size=1000,
    augmentation_config=aug_config
)

# 创建结果保存目录
results_dir = os.path.join(export_path, 'results')
os.makedirs(results_dir, exist_ok=True)

# 训练所有类别
valid_labels = sampler.valid_labels
print(f"开始为{len(valid_labels)}个有效标签训练二分类模型...")

# 保存总体结果
overall_results = {
    'label_id': [],
    'accuracy': [],
    'precision': [],
    'recall': [],
    'f1': [],
    'train_time': [],
    'voxel_count': [],
    'best_grid': []
}

# 定义训练单个标签的函数
def train_binary_model_for_label(label_id, save_prefix="kan_brain_binary"):
    print(f"\n{'='*50}")
    print(f"训练标签 {label_id} 的二分类模型")
    print(f"{'='*50}")
    
    # 获取该标签的数据集
    dataset = sampler.get_dataset_for_label(label_id)
    
    # 检查数据是否足够
    if dataset['train_data'].shape[0] < 100:
        print(f"警告: 标签 {label_id} 的训练样本太少 ({dataset['train_data'].shape[0]}), 跳过训练")
        return None, None
    
    # 创建KAN模型
    model = fastkan.FastKAN(
        layers_hidden=network_structure,
        num_grids=grid_sequence[0]
    ).to(device)
    
    print(f"创建KAN模型: {network_structure}, 初始网格大小: {grid_sequence[0]}")
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"参数数量: {total_params:,}")
    
    # 准备追踪变量
    train_losses = []
    test_losses = []
    train_metrics = []
    test_metrics = []
    used_grid_sizes = []
    
    # 准备用于绘图的实时显示
    plt.figure(figsize=(15, 5))
    plt.ion()  # 打开交互模式
    
    # 保存阶段边界点，用于后续可视化
    phase_boundaries = []
    current_step_count = 0
    start_time = time.time()
    
    # 循环训练每个网格阶段
    for phase_idx in range(3):  # 分为3个主要阶段
        print(f"\n==== 主阶段 {phase_idx+1}/3 ====")
        phase_grid_seq = phase_grid_sequences[phase_idx]
        steps_per_grid = steps_per_phase[phase_idx]
        
        # 根据阶段调整学习率
        if phase_idx == 0:
            current_lr = learning_rate       # 第1阶段使用标准学习率
        elif phase_idx == 1:
            current_lr = learning_rate / 2   # 第2阶段减半学习率
        else:
            current_lr = learning_rate / 5   # 第3阶段使用更小学习率
        
        print(f"当前阶段学习率基准值: {current_lr}")
        
        # 循环训练每个网格点
        for grid_idx, current_grid in enumerate(phase_grid_seq):
            # 如果不是第一个网格点，进行grid扩展
            if not (phase_idx == 0 and grid_idx == 0):
                previous_grid = get_grid_size(model)  # 获取当前模型的网格大小
                if previous_grid != current_grid:  # 只有当网格大小变化时才进行扩展
                    print(f"进行Grid扩展: {previous_grid} -> {current_grid}")
                    try:
                        # 为网格扩展使用小批量数据
                        X_sample, _ = get_train_batch(dataset, 100)  # 只需要特征，不需要标签
                        
                        # 创建一个新的模型，使用新的网格大小
                        old_model = model  # 保存旧模型引用
                        
                        # 创建新模型
                        model = fastkan.FastKAN(
                            layers_hidden=network_structure,
                            num_grids=current_grid
                        ).to(device)
                        
                        # 传输基础层权重（非网格相关部分）
                        for i, (old_layer, new_layer) in enumerate(zip(old_model.layers, model.layers)):
                            if hasattr(old_layer, 'base_linear') and hasattr(new_layer, 'base_linear'):
                                new_layer.base_linear.weight.data.copy_(old_layer.base_linear.weight.data)
                                if hasattr(old_layer.base_linear, 'bias') and old_layer.base_linear.bias is not None:
                                    new_layer.base_linear.bias.data.copy_(old_layer.base_linear.bias.data)
                        
                        print(f"网格扩展成功! 从 {previous_grid} 到 {current_grid}")
                    except Exception as e:
                        print(f"网格扩展失败: {str(e)}")
                        import traceback
                        traceback.print_exc()
            
            # 记录阶段边界，用于后续可视化
            if grid_idx == 0 and phase_idx > 0:
                phase_boundaries.append(current_step_count)
                
            print(f"\n-- 网格点 {grid_idx+1}/{len(phase_grid_seq)} (G={current_grid}), 训练步数: {steps_per_grid} --")
            
            try:
                # 使用AdamW优化器
                optimizer = torch.optim.AdamW(model.parameters(), lr=current_lr, weight_decay=5e-4)
                
                # 添加学习率调度器
                gamma = 0.95 if steps_per_grid > 50 else 0.97
                scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
                
                # 使用二分类损失函数
                criterion = torch.nn.BCEWithLogitsLoss()
                
                # 自定义训练循环
                for step in range(steps_per_grid):
                    if step % 5 == 0 or step == steps_per_grid - 1:
                        print(f"Grid {current_grid}: {step+1}/{steps_per_grid} ({(step+1)/steps_per_grid*100:.1f}%)")
                    
                    # 获取训练批次
                    X_batch, y_batch = get_train_batch(dataset, batch_size)
                    
                    # 前向传播
                    optimizer.zero_grad()
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)
                    
                    # 反向传播
                    loss.backward()
                    
                    # 梯度裁剪
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    
                    # 优化器步骤
                    optimizer.step()
                    
                    # 学习率调度器步骤
                    scheduler.step()
                    
                    # 每5步评估一次性能
                    if step % 5 == 0 or step == steps_per_grid - 1:
                        with torch.no_grad():
                            # 计算训练性能
                            train_preds = torch.sigmoid(outputs)
                            train_metric = calculate_binary_metrics(y_batch.cpu(), train_preds.cpu())
                            
                            # 计算测试性能
                            test_batch_input = dataset['test_input'][:100]
                            test_batch_label = dataset['test_label'][:100]
                            test_outputs = model(test_batch_input)
                            test_preds = torch.sigmoid(test_outputs)
                            test_loss = criterion(test_outputs, test_batch_label).item()
                            test_metric = calculate_binary_metrics(test_batch_label.cpu(), test_preds.cpu())
                            
                            # 记录
                            train_losses.append(loss.item())
                            test_losses.append(test_loss)
                            train_metrics.append(train_metric)
                            test_metrics.append(test_metric)
                            used_grid_sizes.append(current_grid)
                            
                            # 输出当前指标
                            print(f"训练损失: {loss.item():.6f}, 测试损失: {test_loss:.6f}")
                            print(f"训练准确率: {train_metric['accuracy']:.4f}, 测试准确率: {test_metric['accuracy']:.4f}")
                            print(f"训练F1: {train_metric['f1']:.4f}, 测试F1: {test_metric['f1']:.4f}")
                
                # 更新步数计数器
                current_step_count += steps_per_grid
                
                # 更新并显示训练进度图
                plt.clf()
                plt.subplot(1, 2, 1)
                plt.semilogy(train_losses, label='Train Loss')
                plt.semilogy(test_losses, label='Test Loss')
                
                # 在图上标记阶段分界点
                for boundary in phase_boundaries:
                    if boundary < len(train_losses):
                        plt.axvline(x=boundary, color='r', linestyle='--')
                
                plt.xlabel('Steps')
                plt.ylabel('Loss (log scale)')
                plt.legend()
                plt.grid(True)

                plt.subplot(1, 2, 2)
                train_accs = [m['accuracy'] for m in train_metrics]
                test_accs = [m['accuracy'] for m in test_metrics]
                plt.plot(train_accs, label='Train Accuracy')
                plt.plot(test_accs, label='Test Accuracy')
                
                # 在图上标记阶段分界点
                for boundary in phase_boundaries:
                    if boundary < len(train_accs):
                        plt.axvline(x=boundary, color='r', linestyle='--')
                
                plt.xlabel('Steps')
                plt.ylabel('Accuracy')
                plt.legend()
                plt.grid(True)

                plt.suptitle(f'KAN Binary Training (Label {label_id}, Phase {phase_idx+1}/3, Grid={current_grid})')
                plt.tight_layout()
                plt.draw()
                plt.pause(0.1)
                
                # 每个阶段结束时或者每10个网格点保存一次模型
                if grid_idx == len(phase_grid_seq) - 1 or grid_idx % 10 == 0:
                    try:
                        model_folder_base = os.path.join(export_path, 
                                                        f'{save_prefix}_label{label_id}_phase{phase_idx+1}_grid{current_grid}')
                        save_kan_model_folder(model, model_folder_base, add_timestamp=True, grid_size=current_grid)
                        print(f"阶段 {phase_idx+1} 网格点 G={current_grid} 模型已保存")
                    except Exception as e:
                        print(f"保存模型失败: {str(e)}")
                        
            except Exception as e:
                print(f"训练网格点 G={current_grid} 失败: {str(e)}")
                import traceback
                traceback.print_exc()
                
    # 关闭交互模式
    plt.ioff()
    
    # 计算总训练时间
    train_time = time.time() - start_time
    print(f"\n训练完成！总用时: {train_time:.2f} 秒")
    
    # 最终评估
    final_metrics = test_metrics[-1] if test_metrics else None
    if final_metrics:
        print(f"最终测试准确率: {final_metrics['accuracy']:.4f}")
        print(f"最终测试F1分数: {final_metrics['f1']:.4f}")
        print(f"最终测试精确度: {final_metrics['precision']:.4f}")
        print(f"最终测试召回率: {final_metrics['recall']:.4f}")
    
    # 保存最终模型
    try:
        final_model_path = os.path.join(export_path, f'{save_prefix}_label{label_id}_final')
        save_kan_model_folder(model, final_model_path, add_timestamp=True, grid_size=get_grid_size(model))
        print(f"最终模型已保存")
    except Exception as e:
        print(f"保存最终模型失败: {str(e)}")
    
    # 保存训练曲线图
    plt.figure(figsize=(15, 10))
    
    # 损失曲线
    plt.subplot(2, 2, 1)
    plt.plot(train_losses, label='Train Loss', color='blue')
    plt.plot(test_losses, label='Test Loss', color='red')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Training and Test Loss')
    plt.legend()
    plt.grid(True)
    
    # 准确率曲线
    plt.subplot(2, 2, 2)
    plt.plot(train_accs, label='Train Accuracy', color='blue')
    plt.plot(test_accs, label='Test Accuracy', color='red')
    plt.xlabel('Steps')
    plt.ylabel('Accuracy')
    plt.title('Training and Test Accuracy')
    plt.legend()
    plt.grid(True)
    
    # F1分数曲线
    plt.subplot(2, 2, 3)
    train_f1s = [m['f1'] for m in train_metrics]
    test_f1s = [m['f1'] for m in test_metrics]
    plt.plot(train_f1s, label='Train F1', color='blue')
    plt.plot(test_f1s, label='Test F1', color='red')
    plt.xlabel('Steps')
    plt.ylabel('F1 Score')
    plt.title('Training and Test F1 Score')
    plt.legend()
    plt.grid(True)
    
    # 网格大小变化
    plt.subplot(2, 2, 4)
    grid_step_pairs = [(g, i) for i, g in enumerate(used_grid_sizes)]
    unique_grids = []
    unique_steps = []
    
    prev_grid = None
    for grid, step in grid_step_pairs:
        if grid != prev_grid:
            unique_grids.append(grid)
            unique_steps.append(step)
            prev_grid = grid
    
    plt.plot(unique_steps, unique_grids, 'o-', color='green')
    plt.xlabel('Step')
    plt.ylabel('Grid Size')
    plt.title('Grid Size vs Training Step')
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'training_curves_label{label_id}.png'))
    plt.close()
    
    # 返回训练结果
    training_results = {
        'train_losses': train_losses,
        'test_losses': test_losses,
        'train_metrics': train_metrics,
        'test_metrics': test_metrics,
        'grid_sizes': used_grid_sizes,
        'train_time': train_time,
        'final_metrics': final_metrics
    }
    
    return model, training_results

# 辅助函数：从数据集中获取训练批次
def get_train_batch(dataset, batch_size):
    """从数据集中随机采样训练批次"""
    train_data = dataset['train_data']
    train_label = dataset['train_label']
    
    # 确保批次大小不超过数据集大小
    batch_size = min(batch_size, train_data.shape[0])
    
    # 随机选择索引
    indices = np.random.choice(train_data.shape[0], batch_size, replace=False)
    
    # 提取批次数据
    X_batch = train_data[indices]
    y_batch = train_label[indices]
    
    # 转换为PyTorch张量
    X_batch_tensor = torch.from_numpy(X_batch).float().to(device)
    y_batch_tensor = torch.from_numpy(y_batch).float().to(device)
    
    return X_batch_tensor, y_batch_tensor

# 选择需要训练的标签范围
# 注意：您可以选择训练所有标签或者特定范围的标签
# 全部标签：training_labels = valid_labels
# 前10个标签：training_labels = valid_labels[:10]
# 特定标签：training_labels = [42, 56, 78]
training_labels = valid_labels  # 默认训练所有标签

# 循环训练每个标签
for label_idx, label_id in enumerate(training_labels):
    print(f"\n处理标签 {label_id} ({label_idx+1}/{len(training_labels)})...")
    
    # 训练二分类模型
    model, results = train_binary_model_for_label(label_id)
    
    # 保存训练结果
    if model is not None and results is not None:
        # 保存性能指标
        voxel_count = sampler.label_info[label_id]['count']
        final_metrics = results['final_metrics']
        
        overall_results['label_id'].append(label_id)
        overall_results['voxel_count'].append(voxel_count)
        overall_results['train_time'].append(results['train_time'])
        overall_results['best_grid'].append(results['grid_sizes'][-1])
        
        if final_metrics:
            overall_results['accuracy'].append(final_metrics['accuracy'])
            overall_results['precision'].append(final_metrics['precision'])
            overall_results['recall'].append(final_metrics['recall'])
            overall_results['f1'].append(final_metrics['f1'])
        else:
            overall_results['accuracy'].append(None)
            overall_results['precision'].append(None)
            overall_results['recall'].append(None)
            overall_results['f1'].append(None)
        
        # 保存每隔10个标签就保存一次总体结果
        if (label_idx + 1) % 10 == 0 or label_idx == len(training_labels) - 1:
            results_file = os.path.join(results_dir, 'overall_binary_results.npz')
            np.savez(results_file, **overall_results)
            print(f"总体结果已保存到: {results_file}")

# 训练结束后保存总体结果
results_file = os.path.join(results_dir, 'overall_binary_results.npz')
np.savez(results_file, **overall_results)
print(f"所有训练完成! 总体结果已保存到: {results_file}")

# 生成总体性能报告
try:
    # 创建性能汇总表格
    import pandas as pd
    results_df = pd.DataFrame({
        'label_id': overall_results['label_id'],
        'voxel_count': overall_results['voxel_count'],
        'accuracy': overall_results['accuracy'],
        'f1': overall_results['f1'],
        'precision': overall_results['precision'],
        'recall': overall_results['recall'],
        'train_time': overall_results['train_time'],
        'best_grid': overall_results['best_grid']
    })
    
    # 按F1分数排序
    results_df = results_df.sort_values('f1', ascending=False)
    
    # 保存为CSV
    csv_path = os.path.join(results_dir, 'binary_models_performance.csv')
    results_df.to_csv(csv_path, index=False)
    print(f"性能汇总表已保存到: {csv_path}")
    
    # 显示前10个性能最好的模型
    print("\n性能最好的10个标签:")
    print(results_df.head(10))
    
    # 可视化总体性能分布
    plt.figure(figsize=(15, 10))
    
    # 1. F1分数分布
    plt.subplot(2, 2, 1)
    plt.hist(results_df['f1'].dropna(), bins=20)
    plt.xlabel('F1 Score')
    plt.ylabel('Number of Labels')
    plt.title('F1 Score Distribution')
    plt.grid(True)
    
    # 2. 准确率分布
    plt.subplot(2, 2, 2)
    plt.hist(results_df['accuracy'].dropna(), bins=20)
    plt.xlabel('Accuracy')
    plt.ylabel('Number of Labels')
    plt.title('Accuracy Distribution')
    plt.grid(True)
    
    # 3. 体素数量与性能关系
    plt.subplot(2, 2, 3)
    plt.scatter(results_df['voxel_count'], results_df['f1'], alpha=0.7)
    plt.xscale('log')
    plt.xlabel('Voxel Count (log scale)')
    plt.ylabel('F1 Score')
    plt.title('Voxel Count vs F1 Score')
    plt.grid(True)
    
    # 4. 训练时间与性能关系
    plt.subplot(2, 2, 4)
    plt.scatter(results_df['train_time'], results_df['f1'], alpha=0.7)
    plt.xlabel('Training Time (seconds)')
    plt.ylabel('F1 Score')
    plt.title('Training Time vs F1 Score')
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'overall_performance_summary.png'))
    plt.close()
    
    print(f"性能汇总图已保存")
    
except Exception as e:
    print(f"生成性能报告时出错: {str(e)}")
    import traceback
    traceback.print_exc()

初始化平衡采样器...


NameError: name 'AugmentationConfig' is not defined

In [ ]:
import os
import json
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# 模型保存和加载函数 - 修改为支持二分类并包含label_id

def save_kan_model_folder(model, folder_path, add_timestamp=True, grid_size=None, label_id=None):
    """
    保存KAN模型到一个文件夹中，可选添加时间戳和标签ID
    
    参数:
    model: KAN模型实例
    folder_path: 保存文件夹的基础路径
    add_timestamp: 是否添加时间戳到文件夹名称
    grid_size: 当前网格大小（可选）
    label_id: 当前训练的标签ID（可选）
    """
    try:
        # 添加时间戳到文件夹名称
        if add_timestamp:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            if grid_size is not None and label_id is not None:
                folder_path = f"{folder_path}_label{label_id}_grid{grid_size}_{timestamp}"
            elif grid_size is not None:
                folder_path = f"{folder_path}_grid{grid_size}_{timestamp}"
            elif label_id is not None:
                folder_path = f"{folder_path}_label{label_id}_{timestamp}"
            else:
                folder_path = f"{folder_path}_{timestamp}"
        
        # 创建目录
        os.makedirs(folder_path, exist_ok=True)
        
        # 保存模型结构信息
        config = {
            'width': model.width if hasattr(model, 'width') else None,
            'grid': model.grid if hasattr(model, 'grid') else None,
            'k': model.k if hasattr(model, 'k') else 3,
            'saved_at': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'label_id': label_id,  # 保存标签ID
            'binary_mode': True  # 标记为二分类模型
        }
        
        # 保存配置
        with open(os.path.join(folder_path, 'config.json'), 'w') as f:
            json.dump(config, f, indent=4)
        
        # 保存权重
        torch.save(model.state_dict(), os.path.join(folder_path, 'weights.pt'))
        
        # 创建一个简单的README文件
        with open(os.path.join(folder_path, 'README.txt'), 'w') as f:
            f.write(f"KAN Binary Model saved at: {config['saved_at']}\n")
            f.write(f"Architecture: {config['width']}\n")
            f.write(f"Grid Size: {config['grid']}\n")
            f.write(f"K value: {config['k']}\n")
            f.write(f"Label ID: {config['label_id']}\n")
            f.write(f"Mode: Binary Classification\n")
        
        print(f"模型成功保存到文件夹: {folder_path}")
        return folder_path  # 返回实际保存的路径（包含时间戳）
    except Exception as e:
        print(f"保存模型失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

def load_kan_model_folder(folder_path, device=None, binary=True):
    """
    从文件夹加载KAN模型 - 支持二分类模型
    
    参数:
    folder_path: 模型文件夹路径
    device: 计算设备
    binary: 是否为二分类模型
    
    返回:
    model: 加载的KAN模型
    config: 模型配置
    """
    try:
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            
        # 加载配置
        with open(os.path.join(folder_path, 'config.json'), 'r') as f:
            config = json.load(f)
            
        # 创建模型
        # 注意：确保网络结构与保存时一致
        from fastkan import FastKAN
        model = FastKAN(
            layers_hidden=config.get('width', [341, 512, 256, 1]) if binary else [341, 512, 256, 102],
            num_grids=config.get('grid', 3)
        )
        
        # 加载权重
        weights_path = os.path.join(folder_path, 'weights.pt')
        model.load_state_dict(torch.load(weights_path, map_location=device))
        model = model.to(device)
        
        saved_at = config.get('saved_at', 'Unknown')
        label_id = config.get('label_id', 'Unknown')
        print(f"加载模型成功! 模型保存时间: {saved_at}")
        print(f"模型标签ID: {label_id}, 网格大小: {config.get('grid', 'Unknown')}")
        
        return model, config
    except Exception as e:
        print(f"加载模型失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None

def find_latest_model(base_folder, label_id=None):
    """
    查找指定标签ID的最新模型文件夹
    
    参数:
    base_folder: 基础文件夹路径
    label_id: 标签ID，如果为None则查找所有最新模型
    
    返回:
    最新模型的完整路径，如果没有找到则返回None
    """
    try:
        # 获取所有匹配前缀的文件夹
        if label_id is not None:
            matching_folders = [f for f in os.listdir(base_folder) 
                               if os.path.isdir(os.path.join(base_folder, f)) 
                               and f"_label{label_id}_" in f]
        else:
            matching_folders = [f for f in os.listdir(base_folder) 
                               if os.path.isdir(os.path.join(base_folder, f))]
        
        if not matching_folders:
            if label_id is not None:
                print(f"未找到标签ID '{label_id}' 的模型文件夹")
            else:
                print(f"未找到任何模型文件夹")
            return None
        
        # 按时间戳排序（假设格式为prefix_YYYYMMDD_HHMMSS）
        sorted_folders = sorted(matching_folders, reverse=True)
        latest_folder = sorted_folders[0]
        
        full_path = os.path.join(base_folder, latest_folder)
        print(f"找到最新的模型文件夹: {latest_folder}")
        
        return full_path
    except Exception as e:
        print(f"查找最新模型时出错: {str(e)}")
        return None

In [ ]:
# 二分类模型预测函数

def predict_with_binary_kan(model, test_data, batch_size=100, threshold=0.5):
    """
    使用二分类KAN模型分批预测，返回概率值和二分类结果
    
    参数:
    model: 训练好的KAN二分类模型
    test_data: 测试数据，numpy数组
    batch_size: 批处理大小
    threshold: 二分类阈值
    
    返回:
    predictions: 预测概率值
    binary_predictions: 二分类结果（0或1）
    """
    n_samples = test_data.shape[0]
    n_batches = (n_samples + batch_size - 1) // batch_size
    all_predictions = []
    
    print(f"预测{n_samples}个样本，分{n_batches}批处理...")
    
    for i in range(n_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, n_samples)
        
        # 处理当前批次
        with torch.no_grad():
            batch_data = torch.from_numpy(test_data[start_idx:end_idx]).float().to(device)
            logits = model(batch_data)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_predictions.append(probs)
            
            if (i+1) % 5 == 0 or i == n_batches-1:
                print(f"已预测 {end_idx}/{n_samples} 样本")
        
        # 释放内存
        del batch_data
        torch.cuda.empty_cache()
    
    # 合并所有预测结果
    predictions = np.vstack(all_predictions)
    
    # 应用阈值获取二分类结果
    binary_predictions = (predictions > threshold).astype(np.float32)
    
    return predictions, binary_predictions

# 统一评估多个模型的函数
def evaluate_all_binary_models(model_dir, test_data, true_labels=None, batch_size=100):
    """
    评估文件夹中所有二分类模型的性能
    
    参数:
    model_dir: 模型文件夹路径
    test_data: 测试数据
    true_labels: 真实标签（可选）
    batch_size: 批处理大小
    
    返回:
    results: 评估结果字典
    """
    # 查找所有模型文件夹
    model_folders = [os.path.join(model_dir, f) for f in os.listdir(model_dir) 
                    if os.path.isdir(os.path.join(model_dir, f)) and 'final' in f]
    
    if not model_folders:
        print("未找到模型文件夹")
        return None
    
    print(f"找到 {len(model_folders)} 个模型文件夹")
    
    results = {
        'label_id': [],
        'accuracy': [],
        'precision': [],
        'recall': [],
        'f1': [],
        'prediction_time': [],
        'grid_size': []
    }
    
    all_predictions = {}
    
    for folder in tqdm(model_folders, desc="评估模型"):
        try:
            # 加载模型
            model, config = load_kan_model_folder(folder)
            
            if model is None:
                print(f"跳过文件夹 {folder}，无法加载模型")
                continue
            
            label_id = config.get('label_id')
            if label_id is None:
                # 尝试从文件夹名称中提取标签ID
                import re
                match = re.search(r'label(\d+)', os.path.basename(folder))
                if match:
                    label_id = int(match.group(1))
                else:
                    print(f"无法确定文件夹 {folder} 的标签ID，跳过")
                    continue
            
            # 预测
            start_time = time.time()
            probs, binary_preds = predict_with_binary_kan(model, test_data, batch_size)
            pred_time = time.time() - start_time
            
            # 保存预测结果
            all_predictions[label_id] = {
                'probabilities': probs,
                'binary': binary_preds
            }
            
            # 如果有真实标签，计算性能指标
            if true_labels is not None and label_id < true_labels.shape[1]:
                true_binary = true_labels[:, label_id]
                metrics = calculate_binary_metrics(true_binary, probs)
                
                results['label_id'].append(label_id)
                results['accuracy'].append(metrics['accuracy'])
                results['precision'].append(metrics['precision'])
                results['recall'].append(metrics['recall'])
                results['f1'].append(metrics['f1'])
                results['prediction_time'].append(pred_time)
                results['grid_size'].append(config.get('grid', 0))
                
                print(f"标签 {label_id}: 准确率={metrics['accuracy']:.4f}, F1={metrics['f1']:.4f}")
            else:
                print(f"标签 {label_id}: 已生成预测，无法计算指标（缺少真实标签）")
                
        except Exception as e:
            print(f"处理文件夹 {folder} 时出错: {str(e)}")
            import traceback
            traceback.print_exc()
    
    # 保存所有预测结果
    predictions_path = os.path.join(model_dir, 'all_binary_predictions.npz')
    np.savez(predictions_path, **all_predictions)
    print(f"所有预测结果已保存到: {predictions_path}")
    
    # 如果有性能指标，创建汇总报告
    if results['label_id'] and true_labels is not None:
        # 保存性能指标
        results_path = os.path.join(model_dir, 'binary_evaluation_results.npz')
        np.savez(results_path, **results)
        
        # 创建性能汇总图
        plt.figure(figsize=(15, 10))
        
        # F1分数分布
        plt.subplot(2, 2, 1)
        plt.hist(results['f1'], bins=20)
        plt.xlabel('F1 Score')
        plt.ylabel('Number of Models')
        plt.title('F1 Score Distribution')
        plt.grid(True)
        
        # 按体素标签排序的F1分数
        sorted_indices = np.argsort(results['label_id'])
        plt.subplot(2, 2, 2)
        plt.bar(range(len(sorted_indices)), [results['f1'][i] for i in sorted_indices])
        plt.xlabel('Label Index (sorted)')
        plt.ylabel('F1 Score')
        plt.title('F1 Score by Label Index')
        plt.grid(True)
        
        # 准确率vs标签
        plt.subplot(2, 2, 3)
        plt.scatter(results['label_id'], results['accuracy'], alpha=0.7)
        plt.xlabel('Label ID')
        plt.ylabel('Accuracy')
        plt.title('Accuracy by Label ID')
        plt.grid(True)
        
        # 网格大小vs F1分数
        plt.subplot(2, 2, 4)
        plt.scatter(results['grid_size'], results['f1'], alpha=0.7)
        plt.xlabel('Grid Size')
        plt.ylabel('F1 Score')
        plt.title('Grid Size vs F1 Score')
        plt.grid(True)
        
        plt.tight_layout()
        plt.savefig(os.path.join(model_dir, 'binary_evaluation_summary.png'))
        plt.close()
        
        print(f"评估汇总报告已保存")
    
    return results, all_predictions

In [ ]:
# 使用训练好的二分类模型进行推理

def predict_regions_with_binary_models(model_dir, input_data, threshold=0.5):
    """
    使用所有训练好的二分类模型对输入数据进行区域预测
    
    参数:
    model_dir: 模型目录
    input_data: 输入数据，形状为(n_samples, n_features)
    threshold: 二分类阈值
    
    返回:
    predicted_regions: 预测的区域标签，形状为(n_samples, n_classes)
    probabilities: 各区域的预测概率
    """
    # 查找所有最终模型
    model_folders = [os.path.join(model_dir, f) for f in os.listdir(model_dir) 
                    if os.path.isdir(os.path.join(model_dir, f)) and 'final' in f]
    
    if not model_folders:
        print("未找到模型文件夹")
        return None, None
    
    print(f"找到 {len(model_folders)} 个模型文件夹")
    
    # 获取样本数量和特征维度
    n_samples = input_data.shape[0]
    
    # 确定最大标签ID
    max_label_id = 0
    for folder in model_folders:
        try:
            _, config = load_kan_model_folder(folder)
            if config and 'label_id' in config:
                max_label_id = max(max_label_id, config['label_id'])
            else:
                # 尝试从文件夹名称中提取标签ID
                import re
                match = re.search(r'label(\d+)', os.path.basename(folder))
                if match:
                    max_label_id = max(max_label_id, int(match.group(1)))
        except:
            pass
    
    n_classes = max_label_id + 1
    print(f"最大标签ID: {max_label_id}, 总类别数: {n_classes}")
    
    # 初始化结果数组
    probabilities = np.zeros((n_samples, n_classes))
    
    # 对每个模型进行预测
    for folder in tqdm(model_folders, desc="应用模型"):
        try:
            # 加载模型
            model, config = load_kan_model_folder(folder)
            
            if model is None:
                continue
            
            label_id = config.get('label_id')
            if label_id is None:
                # 尝试从文件夹名称中提取标签ID
                import re
                match = re.search(r'label(\d+)', os.path.basename(folder))
                if match:
                    label_id = int(match.group(1))
                else:
                    continue
            
            # 确保标签ID有效
            if label_id >= n_classes:
                print(f"警告：标签ID {label_id} 超出范围，最大应为 {n_classes-1}")
                continue
            
            # 预测
            probs, _ = predict_with_binary_kan(model, input_data, batch_size=100)
            
            # 保存概率值
            probabilities[:, label_id] = probs.flatten()
            
            print(f"标签 {label_id} 预测完成，概率值范围: [{np.min(probs):.4f}, {np.max(probs):.4f}]")
            
        except Exception as e:
            print(f"处理文件夹 {folder} 时出错: {str(e)}")
            continue
    
    # 应用阈值，获取二分类预测结果
    binary_predictions = (probabilities > threshold).astype(np.float32)
    
    # 对每个样本，找出概率最高的类别作为区域预测
    predicted_regions = np.argmax(probabilities, axis=1)
    
    # 检查是否有样本没有任何区域预测为1
    no_prediction_mask = np.sum(binary_predictions, axis=1) == 0
    if np.any(no_prediction_mask):
        n_no_pred = np.sum(no_prediction_mask)
        print(f"警告: {n_no_pred} 个样本 ({n_no_pred/n_samples*100:.2f}%) 没有任何区域预测为1")
        # 对于这些样本，使用概率最高的区域作为预测
        for i in np.where(no_prediction_mask)[0]:
            max_prob_region = np.argmax(probabilities[i])
            binary_predictions[i, max_prob_region] = 1
    
    # 检查是否有样本有多个区域预测为1
    multi_prediction_mask = np.sum(binary_predictions, axis=1) > 1
    if np.any(multi_prediction_mask):
        n_multi_pred = np.sum(multi_prediction_mask)
        print(f"注意: {n_multi_pred} 个样本 ({n_multi_pred/n_samples*100:.2f}%) 有多个区域预测为1")
        # 这是正常的，因为一个体素可能属于多个区域
    
    return predicted_regions, probabilities, binary_predictions

In [ ]:
# 评估二分类模型性能

# 加载验证数据
print("加载验证数据...")
val_data = np.load(f"{output_path}/val_data.npy")
val_label = np.load(f"{output_path}/val_label.npy")
print(f"验证数据: {val_data.shape}, 标签: {val_label.shape}")

# 设置评估参数
test_size = 1000  # 限制评估样本数量以提高速度
model_dir = export_path  # 二分类模型目录

# 如果验证集太大，随机选择一部分
if val_data.shape[0] > test_size:
    indices = np.random.choice(val_data.shape[0], test_size, replace=False)
    test_data = val_data[indices]
    test_labels = val_label[indices]
else:
    test_data = val_data
    test_labels = val_label

print(f"使用 {test_data.shape[0]} 个样本进行评估")

# 评估所有模型
results, all_predictions = evaluate_all_binary_models(
    model_dir=model_dir,
    test_data=test_data,
    true_labels=test_labels,
    batch_size=100
)

# 计算overall性能
if results:
    # 计算微平均指标
    avg_accuracy = np.mean(results['accuracy'])
    avg_precision = np.mean(results['precision'])
    avg_recall = np.mean(results['recall'])
    avg_f1 = np.mean(results['f1'])
    
    print("\n总体性能指标 (宏平均):")
    print(f"平均准确率: {avg_accuracy:.4f}")
    print(f"平均精确度: {avg_precision:.4f}")
    print(f"平均召回率: {avg_recall:.4f}")
    print(f"平均F1分数: {avg_f1:.4f}")
    
    # 找出性能最好和最差的类别
    best_idx = np.argmax(results['f1'])
    worst_idx = np.argmin(results['f1'])
    
    best_label = results['label_id'][best_idx]
    worst_label = results['label_id'][worst_idx]
    
    print(f"\n性能最好的类别: 标签 {best_label}")
    print(f"  F1: {results['f1'][best_idx]:.4f}")
    print(f"  准确率: {results['accuracy'][best_idx]:.4f}")
    print(f"  精确度: {results['precision'][best_idx]:.4f}")
    print(f"  召回率: {results['recall'][best_idx]:.4f}")
    
    print(f"\n性能最差的类别: 标签 {worst_label}")
    print(f"  F1: {results['f1'][worst_idx]:.4f}")
    print(f"  准确率: {results['accuracy'][worst_idx]:.4f}")
    print(f"  精确度: {results['precision'][worst_idx]:.4f}")
    print(f"  召回率: {results['recall'][worst_idx]:.4f}")
    
    # 创建性能分布饼图
    f1_bins = [0, 0.3, 0.5, 0.7, 0.9, 1.0]
    f1_counts = [sum((f1 >= low) & (f1 < high) for f1 in results['f1']) 
                 for low, high in zip(f1_bins[:-1], f1_bins[1:])]
    
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.pie(f1_counts, labels=[f"{low:.1f}-{high:.1f}" for low, high in zip(f1_bins[:-1], f1_bins[1:])],
           autopct='%1.1f%%', startangle=90)
    plt.title('F1分数分布')
    
    # 创建标签计数柱状图
    plt.subplot(1, 2, 2)
    
    # 获取标签体素计数信息
    label_counts = []
    for label_id in results['label_id']:
        try:
            count = sampler.label_info[label_id]['count']
            label_counts.append(count)
        except:
            label_counts.append(0)
    
    # 计算性能与标签体素数量的相关性
    from scipy.stats import pearsonr
    corr, p_value = pearsonr(label_counts, results['f1'])
    
    plt.scatter(label_counts, results['f1'], alpha=0.7)
    plt.xscale('log')
    plt.xlabel('体素数量 (对数尺度)')
    plt.ylabel('F1分数')
    plt.title(f'体素数量与F1分数关系\n相关系数: {corr:.3f} (p={p_value:.3f})')
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(model_dir, 'performance_distribution.png'))
    plt.show()
    
    print(f"\n体素数量与F1分数的相关系数: {corr:.3f} (p值={p_value:.3f})")
    if p_value < 0.05:
        if corr > 0:
            print("统计显著性结果: 体素数量越多，模型性能越好")
        else:
            print("统计显著性结果: 体素数量越少，模型性能越好")
    else:
        print("统计显著性结果: 体素数量与模型性能无显著相关性")

In [ ]:
# 推理函数 - 可用于新数据预测

def predict_brain_regions(input_data, model_dir=export_path, threshold=0.5, batch_size=100):
    """
    对输入的MRI特征数据预测脑区域
    
    参数:
    input_data: 输入特征数据，形状为(n_samples, 341)
    model_dir: 模型目录
    threshold: 二分类阈值
    batch_size: 批处理大小
    
    返回:
    regions: 预测的脑区域标签
    probabilities: 各区域的预测概率
    binary_predictions: 二分类预测结果
    """
    # 数据预处理
    if isinstance(input_data, list) or isinstance(input_data, tuple):
        input_data = np.array(input_data)
    
    if len(input_data.shape) == 1:
        # 单个样本，扩展为2D
        input_data = input_data.reshape(1, -1)
    
    # 检查特征维度
    if input_data.shape[1] != 341:
        raise ValueError(f"输入特征维度应为341，但收到{input_data.shape[1]}")
    
    # 预测
    regions, probabilities, binary_predictions = predict_regions_with_binary_models(
        model_dir=model_dir,
        input_data=input_data,
        threshold=threshold
    )
    
    return regions, probabilities, binary_predictions

def save_prediction_results(regions, probabilities, binary_predictions, output_file=None):
    """保存预测结果"""
    if output_file is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = f"brain_region_predictions_{timestamp}.npz"
    
    np.savez(
        output_file,
        regions=regions,
        probabilities=probabilities,
        binary_predictions=binary_predictions
    )
    
    print(f"预测结果已保存到: {output_file}")
    return output_file

# 示例：应用模型进行预测
def demo_prediction(test_data_path, model_dir=export_path):
    """运行预测演示"""
    print("加载测试数据...")
    try:
        test_data = np.load(test_data_path)
        print(f"测试数据形状: {test_data.shape}")
        
        # 选择小批量数据进行演示
        n_samples = min(10, test_data.shape[0])
        demo_data = test_data[:n_samples]
        print(f"使用 {n_samples} 个样本进行演示")
        
        # 预测
        print("运行预测...")
        regions, probabilities, binary_predictions = predict_brain_regions(
            input_data=demo_data,
            model_dir=model_dir
        )
        
        # 显示结果
        print("\n预测结果:")
        for i in range(n_samples):
            predicted_region = regions[i]
            region_prob = probabilities[i, predicted_region]
            print(f"样本 {i+1}: 预测脑区域 = {predicted_region}, 置信度 = {region_prob:.4f}")
            
            # 显示所有预测为1的区域
            positive_regions = np.where(binary_predictions[i] == 1)[0]
            if len(positive_regions) > 0:
                print(f"  预测为阳性的所有区域: {positive_regions}")
                print(f"  对应概率: {[probabilities[i, r]:.4f for r in positive_regions]}")
            else:
                print(f"  没有区域预测为阳性")
        
        # 保存结果
        save_prediction_results(regions, probabilities, binary_predictions)
        
        return regions, probabilities, binary_predictions
    
    except Exception as e:
        print(f"演示失败: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, None

In [ ]:
# 可视化预测结果

def visualize_binary_prediction_results(binary_predictions, probabilities, true_labels=None, top_k=10):
    """
    可视化二分类预测结果
    
    参数:
    binary_predictions: 二分类预测结果，形状为(n_samples, n_classes)
    probabilities: 预测概率，形状为(n_samples, n_classes)
    true_labels: 真实标签（可选），形状为(n_samples, n_classes)
    top_k: 显示前k个最常预测为阳性的区域
    """
    plt.figure(figsize=(15, 10))
    
    # 1. 计算每个区域预测为阳性的样本比例
    positive_rates = np.mean(binary_predictions, axis=0)
    
    # 获取预测阳性率最高的top_k个区域
    top_regions = np.argsort(positive_rates)[::-1][:top_k]
    
    # 绘制阳性预测比例条形图
    plt.subplot(2, 2, 1)
    plt.bar(range(len(top_regions)), [positive_rates[i] for i in top_regions])
    plt.xticks(range(len(top_regions)), [f"区域{i}" for i in top_regions], rotation=45)
    plt.xlabel('区域ID')
    plt.ylabel('阳性预测比例')
    plt.title(f'预测阳性率最高的{top_k}个区域')
    plt.grid(True)
    
    # 2. 绘制概率分布直方图
    plt.subplot(2, 2, 2)
    for i, region_id in enumerate(top_regions[:5]):  # 只显示前5个区域的概率分布
        plt.hist(probabilities[:, region_id], bins=20, alpha=0.5, label=f'区域{region_id}')
    plt.xlabel('预测概率')
    plt.ylabel('样本数量')
    plt.title('前5个区域的概率分布')
    plt.legend()
    plt.grid(True)
    
    # 3. 如果有真实标签，计算准确率矩阵
    if true_labels is not None:
        # 计算每个区域的准确率
        accuracies = []
        for i in range(binary_predictions.shape[1]):
            if np.sum(true_labels[:, i]) > 0:  # 确保有该区域的真实标签
                acc = np.mean(binary_predictions[:, i] == true_labels[:, i])
                accuracies.append((i, acc))
        
        # 按准确率排序
        accuracies.sort(key=lambda x: x[1], reverse=True)
        
        # 绘制准确率条形图
        plt.subplot(2, 2, 3)
        region_ids = [x[0] for x in accuracies[:top_k]]
        accs = [x[1] for x in accuracies[:top_k]]
        plt.bar(range(len(region_ids)), accs)
        plt.xticks(range(len(region_ids)), [f"区域{i}" for i in region_ids], rotation=45)
        plt.xlabel('区域ID')
        plt.ylabel('准确率')
        plt.title(f'准确率最高的{top_k}个区域')
        plt.grid(True)
        
        # 4. 绘制混淆矩阵热图
        plt.subplot(2, 2, 4)
        # 为前top_k个区域计算混淆矩阵
        from sklearn.metrics import confusion_matrix
        import seaborn as sns
        
        selected_region = top_regions[0]  # 选择预测阳性率最高的区域
        cm = confusion_matrix(true_labels[:, selected_region], binary_predictions[:, selected_region])
        
        # 标准化混淆矩阵
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
        # 绘制热图
        sns.heatmap(cm_normalized, annot=cm, fmt='d', cmap='Blues', cbar=False)
        plt.xlabel('预测标签')
        plt.ylabel('真实标签')
        plt.title(f'区域{selected_region}的混淆矩阵')
        plt.xticks([0.5, 1.5], ['0', '1'])
        plt.yticks([0.5, 1.5], ['0', '1'])
    
    else:
        # 如果没有真实标签，显示每个样本被预测为阳性的区域数量分布
        positive_counts = np.sum(binary_predictions, axis=1)
        
        plt.subplot(2, 2, 3)
        plt.hist(positive_counts, bins=range(np.max(positive_counts)+2), alpha=0.7)
        plt.xlabel('预测为阳性的区域数量')
        plt.ylabel('样本数量')
        plt.title('样本被预测为阳性的区域数量分布')
        plt.grid(True)
        
        # 计算区域之间的共现矩阵
        n_regions = min(20, binary_predictions.shape[1])  # 限制为前20个区域以避免矩阵过大
        cooccurrence = np.zeros((n_regions, n_regions))
        
        for i in range(n_regions):
            for j in range(n_regions):
                if i == j:
                    cooccurrence[i, j] = positive_rates[i]
                else:
                    # 计算区域i和区域j同时为1的样本比例
                    cooccurrence[i, j] = np.mean(binary_predictions[:, i] & binary_predictions[:, j])
        
        plt.subplot(2, 2, 4)
        sns.heatmap(cooccurrence, cmap='YlGnBu')
        plt.xlabel('区域ID')
        plt.ylabel('区域ID')
        plt.title('区域预测共现矩阵')
        plt.xticks(np.arange(n_regions)+0.5, range(n_regions))
        plt.yticks(np.arange(n_regions)+0.5, range(n_regions))
    
    plt.tight_layout()
    plt.savefig(os.path.join(export_path, 'prediction_visualization.png'))
    plt.show()